# Role table and Betweenness table

In [35]:
import pickle
import pandas as pd
import networkx as nx

In [36]:
l_g_file=['carnival_tourist','no_carnival_tourist','carnival_residents','no_carnival_residents']

## Role table

In [37]:
list_rol=list()
for i in l_g_file:
    with open('../models/'+i+'_rol.pickle', 'rb') as handle:
        list_rol.append(pickle.load(handle))
        
df_roles = pd.DataFrame({
    l_g_file[0]: pd.Series(list_rol[0]),
    l_g_file[1]: pd.Series(list_rol[1]),
    l_g_file[2]: pd.Series(list_rol[2]),
    l_g_file[3]: pd.Series(list_rol[3])
})

df_roles

,carnival_tourist,no_carnival_tourist,carnival_residents,no_carnival_residents
Aeroporto,R2 Peripheral,R2 Peripheral,R2 Peripheral,R2 Peripheral
Alberoni Faro Rocchetta N,R1 Ultra-peripheral,R1 Ultra-peripheral,NaN,R2 Peripheral
Arsenale DX,R3 Non-hub connectors,R3 Non-hub connectors,R2 Peripheral,R3 Non-hub connectors
Bacini - Arsenale Nord,R3 Non-hub connectors,R3 Non-hub connectors,R2 Peripheral,R2 Peripheral
Burano DX per Treporti,R6 Connector hubs,R6 Connector hubs,R6 Connector hubs,R6 Connector hubs
...,...,...,...,...
Tre Archi,R2 Peripheral,R3 Non-hub connectors,R3 Non-hub connectors,R3 Non-hub connectors
Treporti,R2 Peripheral,R2 Peripheral,R2 Peripheral,R2 Peripheral
Tronchetto 'B' SX,R3 Non-hub connectors,R3 Non-hub connectors,R3 Non-hub connectors,R6 Connector hubs
Zattere SX,R3 Non-hub connectors,R3 Non-hub connectors,R6 Connector hubs,R6 Connector hubs


In [38]:
df_roles.to_csv('../models/complete_guimera_cartography.csv')

There are a few stops that completely dissapear from the corresponding graph, meaning that they're not used by the given user profile and time period.

Those stops are the following:

In [39]:
rows_with_nan = df_roles[df_roles.isna().any(axis=1)]
rows_with_nan

,carnival_tourist,no_carnival_tourist,carnival_residents,no_carnival_residents
Alberoni Faro Rocchetta N,R1 Ultra-peripheral,R1 Ultra-peripheral,NaN,R2 Peripheral
Certosa,R2 Peripheral,R2 Peripheral,NaN,NaN
Cimitero S. Michele,R2 Peripheral,R2 Peripheral,NaN,R3 Non-hub connectors
Lido S. Nicolo' M/N,NaN,R2 Peripheral,R2 Peripheral,R2 Peripheral
Pellestrina Cimitero M/N,NaN,R2 Peripheral,NaN,R1 Ultra-peripheral
S. Erasmo Capannone,R2 Peripheral,R2 Peripheral,NaN,R2 Peripheral
S. Erasmo Punta Vela,NaN,R1 Ultra-peripheral,NaN,R1 Ultra-peripheral
S. Lazzaro,NaN,R2 Peripheral,NaN,NaN
S. Servolo,R3 Non-hub connectors,R2 Peripheral,NaN,R2 Peripheral
Torcello,R2 Peripheral,R2 Peripheral,NaN,R2 Peripheral


## Betweenness table

Betweenness Definition: for a node v, betweenness centrality is the sum over all pairs (s,t) of the fraction of shortest paths from s to t that pass through v. A normalized value scales this to [0,1].

Interpretation:
- High betweenness → node acts as a bridge or broker connecting different parts of the network; it can control or facilitate flow between communities.
- Low betweenness → node is peripheral or sits inside a densely connected region (many alternate paths).

In [40]:
list_btw=list()

for i in l_g_file:
    G=nx.read_graphml('../models/'+i+'.graphml')
    
    # compute betweenness for all nodes in G (set weight='weight' if your graph is weighted)
    btw = nx.betweenness_centrality(G, k=None, normalized=True, weight=None, endpoints=False, seed=None)
    list_btw.append(btw)

# build DataFrame aligning nodes as index
df_btw = pd.concat([pd.Series(d, name=f) for d, f in zip(list_btw, l_g_file)], axis=1)
df_btw = df_btw.sort_index()

df_btw

,carnival_tourist,no_carnival_tourist,carnival_residents,no_carnival_residents
Accademia DX,0.000000,0.000000,NaN,0.000000
Aeroporto,0.000376,0.000182,0.000099,0.000090
Alberoni Faro Rocchetta N,0.000000,0.000217,0.000000,0.000038
Arsenale DX,0.010718,0.012010,0.010773,0.006781
Bacini - Arsenale Nord,0.001270,0.016507,0.001330,0.000634
...,...,...,...,...
Tre Archi,0.000271,0.001088,0.001180,0.001426
Treporti,0.000000,0.000037,0.000602,0.002881
Tronchetto 'B' SX,0.016578,0.017163,0.004764,0.024362
Zattere SX,0.007884,0.027114,0.016315,0.014217


In [41]:
#Results are available in CSV form for further analysis if needed
df_btw.to_csv('../models/complete_betweenness.csv')

## Betweenness and Roles by profile

In this following analysis, we match roles and bewteenness for the stops on each graph. 

We can see that in many cases, high betweenness values correspond to R6 role nodes, although this is not always the case. Also, it is interesting that the betweenness changes a bit depending on the carnival period.

In [42]:
# create combined dataframes pairing role and betweenness for each profile
role_btw_dfs = {}

for profile in l_g_file:
    role_s = df_roles[profile] if profile in df_roles.columns else pd.Series(dtype=object)
    btw_s = df_btw[profile] if profile in df_btw.columns else pd.Series(dtype=float)
    combined = pd.concat([role_s.rename('role'), btw_s.rename('betweenness')], axis=1)
    combined = combined.sort_index()
    role_btw_dfs[profile] = combined

In [43]:
# 10 nodes with highest betweenness for Tourists in regular period
role_btw_dfs['no_carnival_tourist'].sort_values(by='betweenness', ascending=False).head(10)

,role,betweenness
"S. Zaccaria (Pieta') ""A""",R6 Connector hubs,0.057413
"F.te Nove ""C""",R2 Peripheral,0.048093
"Murano Colonna ""A""",R6 Connector hubs,0.040472
"Lido (S.M.E.) ""B""",R6 Connector hubs,0.040454
Burano DX per Treporti,R6 Connector hubs,0.028562
"Rialto ""D""",R6 Connector hubs,0.028230
Zattere SX,R3 Non-hub connectors,0.027114
P.le Roma (Hotel S. Chiar,R3 Non-hub connectors,0.019663
"Ferrovia ""B""",R6 Connector hubs,0.018864
S. Marco (Vallaresso) SX,R6 Connector hubs,0.017602


In [44]:
# 10 nodes with highest betweenness for Tourist in Carnival
role_btw_dfs['carnival_tourist'].sort_values(by='betweenness', ascending=False).head(10)

,role,betweenness
"S. Zaccaria (Pieta') ""A""",R6 Connector hubs,0.071255
"Rialto ""D""",R6 Connector hubs,0.047325
"Murano Colonna ""A""",R6 Connector hubs,0.046492
"F.te Nove ""C""",R2 Peripheral,0.045943
Burano DX per Treporti,R6 Connector hubs,0.037721
"Lido (S.M.E.) ""B""",R6 Connector hubs,0.034760
P.le Roma (Hotel S. Chiar,R3 Non-hub connectors,0.029964
"Ferrovia ""B""",R6 Connector hubs,0.029958
Lido bus,R2 Peripheral,0.019590
Tronchetto 'B' SX,R3 Non-hub connectors,0.016578


In [45]:
# 10 nodes with highest betweenness for Residents in regular period
role_btw_dfs['no_carnival_residents'].sort_values(by='betweenness', ascending=False).head(10)

,role,betweenness
"Lido (S.M.E.) ""B""",R6 Connector hubs,0.074635
P.le Roma (Hotel S. Chiar,R2 Peripheral,0.072522
"S. Zaccaria (Pieta') ""A""",R6 Connector hubs,0.060838
"F.te Nove ""C""",R6 Connector hubs,0.049274
"Rialto ""D""",R2 Peripheral,0.045932
Giudecca Palanca SX,R6 Connector hubs,0.031259
S. Elena DX,R2 Peripheral,0.028332
"Murano Colonna ""A""",R6 Connector hubs,0.028087
Burano DX per Treporti,R6 Connector hubs,0.024769
Tronchetto 'B' SX,R6 Connector hubs,0.024362


In [46]:
# 10 nodes with highest betweenness for Residents in Carnival
role_btw_dfs['carnival_residents'].sort_values(by='betweenness', ascending=False).head(10)

,role,betweenness
"Lido (S.M.E.) ""B""",R6 Connector hubs,0.131225
P.le Roma (Hotel S. Chiar,R2 Peripheral,0.101473
"S. Zaccaria (Pieta') ""A""",R6 Connector hubs,0.044943
"Rialto ""D""",R2 Peripheral,0.044168
Burano DX per Treporti,R6 Connector hubs,0.039066
Giudecca Palanca SX,R6 Connector hubs,0.034947
"F.te Nove ""C""",R3 Non-hub connectors,0.032476
Giardini Biennale SX,R3 Non-hub connectors,0.024440
"Murano Colonna ""A""",R6 Connector hubs,0.024261
Piazzale Roma,R3 Non-hub connectors,0.021270
